In [16]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
ORDERS = "orders"
CUSTOMERS = "customers"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/"

In [17]:
spark = (
        SparkSession.builder.appName("test_orders")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

In [18]:
orders = spark.read.format("delta").load(CLEANED_PATH + ORDERS)

# get customer id with the biggest number of orders
customer_id = orders.groupBy("customer_id").count().orderBy("count", ascending=False).first()[0]
print(customer_id)
filter_expression = f"customer_id = {customer_id}"
orders.orderBy("customer_id", sf.col("created_at").asc()).filter(filter_expression).select("order_id", "customer_id", "created_at").show(10)

641


+--------+-----------+-------------------+
|order_id|customer_id|         created_at|
+--------+-----------+-------------------+
|     296|        641|2026-03-27 22:17:49|
|     295|        641|2026-03-27 22:17:49|
|     910|        641|2026-03-27 22:17:49|
|     282|        641|2026-03-27 22:17:49|
|    1292|        641|2026-03-27 22:17:58|
|    1908|        641|2026-03-27 22:17:58|
|    1293|        641|2026-03-27 22:17:58|
|    1278|        641|2026-03-27 22:17:58|
|    1707|        641|2026-03-27 22:17:58|
|    2296|        641|2026-03-27 22:36:50|
+--------+-----------+-------------------+
only showing top 10 rows



In [19]:
orders = spark.read.format("delta").load(CURATED_PATH + ORDERS)
customers = spark.read.format("delta").load(CURATED_PATH + CUSTOMERS)

customers.orderBy("customer_id", sf.col("created_at").asc()).filter(filter_expression).select("customer_id", "created_at", "sk_customer", "effective_from", "effective_to", "is_active").show(10)

orders.orderBy("customer_id", sf.col("created_at").asc()).filter(filter_expression).select("order_id", "customer_id", "created_at", "sk_customer").show(10)

+-----------+-------------------+-----------+-------------------+-------------------+---------+
|customer_id|         created_at|sk_customer|     effective_from|       effective_to|is_active|
+-----------+-------------------+-----------+-------------------+-------------------+---------+
|        641|2026-03-27 22:17:46|       1281|2026-03-27 22:17:46|2026-03-27 22:17:55|    false|
|        641|2026-03-27 22:17:55|       1282|2026-03-27 22:17:55|2026-03-27 22:36:47|    false|
|        641|2026-03-27 22:36:47|       4562|2026-03-27 22:36:47|2026-03-27 22:36:55|    false|
|        641|2026-03-27 22:36:55|       4563|2026-03-27 22:36:55|2026-03-27 22:37:03|    false|
|        641|2026-03-27 22:37:03|       4564|2026-03-27 22:37:03|               NULL|     true|
+-----------+-------------------+-----------+-------------------+-------------------+---------+



+--------+-----------+-------------------+-----------+
|order_id|customer_id|         created_at|sk_customer|
+--------+-----------+-------------------+-----------+
|     282|        641|2026-03-27 22:17:49|       1281|
|     910|        641|2026-03-27 22:17:49|       1281|
|     295|        641|2026-03-27 22:17:49|       1281|
|     296|        641|2026-03-27 22:17:49|       1281|
|    1908|        641|2026-03-27 22:17:58|       1282|
|    1292|        641|2026-03-27 22:17:58|       1282|
|    1278|        641|2026-03-27 22:17:58|       1282|
|    1293|        641|2026-03-27 22:17:58|       1282|
|    1707|        641|2026-03-27 22:17:58|       1282|
|    2296|        641|2026-03-27 22:36:50|       4562|
+--------+-----------+-------------------+-----------+
only showing top 10 rows



In [20]:
spark.stop()